# MixLLM 4/8/16 staged gate

This notebook runs deterministic reference, allocation, packing and capability gates. Model quality and throughput are reported only when the requested model and runtime are available; unsupported native backends remain explicit rather than being counted as passes. SM75 measurements are operator microbenchmarks, not model throughput claims.


In [ ]:
import json, os, platform, subprocess, sys, tempfile
from pathlib import Path
import torch
print('Python', sys.version)
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU', torch.cuda.get_device_name(0))
    print('Capability', torch.cuda.get_device_capability(0))


In [ ]:
sources = {'mixllm/__init__.py': 'import sysconfig\nfrom pathlib import Path\n\n\ndef load_cuda_extension() -> None:\n    """Load the compiled extension only when the CUDA runtime is requested."""\n    import torch\n\n    current_dir = Path(__file__).resolve().parent\n    ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")\n    extension = current_dir / f"kernels{ext_suffix}"\n    if not extension.is_file():\n        raise RuntimeError(\n            f"MixLLM CUDA extension is not built: {extension}. "\n            "Reference quantization and capability modules remain available."\n        )\n    torch.ops.load_library(str(extension))\n\n\ndef __getattr__(name):\n    """Preserve the legacy package API without eager CUDA side effects."""\n    if name == "LinearMixLLM":\n        load_cuda_extension()\n        from mixllm.nn.modules.linear import LinearMixLLM\n        return LinearMixLLM\n    if name == "LinearMixLLM4vLLM":\n        load_cuda_extension()\n        from mixllm.nn.modules.linear_for_vllm import LinearMixLLM4vLLM\n        return LinearMixLLM4vLLM\n    if name == "MixLLMConfig":\n        from mixllm.nn.modules.mixllm_config import MixLLMConfig\n        return MixLLMConfig\n    raise AttributeError(name)\n', 'mixllm/quantization/__init__.py': '', 'mixllm/nn/__init__.py': '', 'mixllm/nn/modules/__init__.py': '', 'mixllm/quantization/three_level.py': '"""Three-level allocation and reference quantization for MixLLM.\n\nThe allocator uses the value of an upgrade (L4-L8 or L8-L16), rather than\nthe absolute loss of the lowest precision.  This keeps the existing global\noutput-feature design while making the extension to FP16 unambiguous.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport json\nfrom pathlib import Path\nfrom typing import Dict, Mapping, Sequence\n\nimport torch\n\n\nLEVELS = (4, 8, 16)\n\n\n@dataclass(frozen=True)\nclass ThreeLevelBudget:\n    """Percentages of output channels assigned to each precision."""\n\n    bit4: int\n    bit8: int\n    bit16: int\n\n    def __post_init__(self) -> None:\n        values = (self.bit4, self.bit8, self.bit16)\n        if any(value < 0 for value in values) or sum(values) != 100:\n            raise ValueError("bit4 + bit8 + bit16 must equal 100")\n\n    def as_dict(self) -> Dict[int, int]:\n        return {4: self.bit4, 8: self.bit8, 16: self.bit16}\n\n\n@dataclass(frozen=True)\nclass ThreeLevelAllocation:\n    """A complete, deterministic output-channel allocation."""\n\n    indices: Dict[int, tuple[int, ...]]\n    scores: Dict[int, tuple[float, ...]]\n    budget: ThreeLevelBudget\n    version: int = 1\n    metadata: Dict[str, object] = field(default_factory=dict)\n\n    def verify(self, out_features: int) -> None:\n        assigned = [index for bit in LEVELS for index in self.indices[bit]]\n        if sorted(assigned) != list(range(out_features)):\n            raise ValueError("allocation must cover every output channel exactly once")\n        if any(index < 0 or index >= out_features for index in assigned):\n            raise ValueError("allocation contains an out-of-range channel")\n\n    def to_json(self, path: str | Path) -> None:\n        payload = {\n            "format": "mixllm-three-level",\n            "version": self.version,\n            "metadata": self.metadata,\n            "budget": {str(k): v for k, v in self.budget.as_dict().items()},\n            "indices": {str(k): list(v) for k, v in self.indices.items()},\n            "scores": {str(k): list(v) for k, v in self.scores.items()},\n        }\n        Path(path).write_text(json.dumps(payload, indent=2) + "\\n", encoding="utf-8")\n\n    @classmethod\n    def from_json(cls, path: str | Path) -> "ThreeLevelAllocation":\n        payload = json.loads(Path(path).read_text(encoding="utf-8"))\n        if payload.get("format") != "mixllm-three-level":\n            raise ValueError("unsupported allocation format")\n        budget = ThreeLevelBudget(**{\n            f"bit{bit}": int(value) for bit, value in payload["budget"].items()\n        })\n        allocation = cls(\n            indices={int(bit): tuple(int(i) for i in values)\n                     for bit, values in payload["indices"].items()},\n            scores={int(bit): tuple(float(i) for i in values)\n                    for bit, values in payload["scores"].items()},\n            budget=budget,\n            version=int(payload["version"]),\n            metadata=dict(payload.get("metadata", {})),\n        )\n        if set(allocation.indices) != set(LEVELS):\n            raise ValueError("allocation must contain 4, 8 and 16 bit partitions")\n        return allocation\n\n\ndef _rounded_counts(out_features: int, budget: ThreeLevelBudget,\n                    alignment: int) -> Dict[int, int]:\n    """Round counts to alignment while preserving the requested total."""\n    raw = {bit: out_features * pct / 100 for bit, pct in budget.as_dict().items()}\n    counts = {bit: int(raw[bit] // alignment) * alignment for bit in LEVELS}\n    remainder = out_features - sum(counts.values())\n    order = sorted(LEVELS, key=lambda bit: (raw[bit] - counts[bit], -bit), reverse=True)\n    for bit in order:\n        if remainder < alignment:\n            break\n        counts[bit] += alignment\n        remainder -= alignment\n    # A tail is legal for the metadata/reference path; the backend may reject it.\n    counts[order[0]] += remainder\n    return counts\n\n\ndef allocate_channels(losses: Mapping[int, torch.Tensor | Sequence[float]],\n                      budget: ThreeLevelBudget, alignment: int = 1\n                      ) -> ThreeLevelAllocation:\n    """Allocate channels using marginal upgrade losses.\n\n    ``losses[bit][channel]`` is the estimated output error when that channel\n    is quantized at ``bit``.  Lower is better.  The allocation is global across\n    all channels and therefore matches MixLLM\'s output-feature abstraction.\n    """\n    if alignment < 1:\n        raise ValueError("alignment must be positive")\n    values = {bit: torch.as_tensor(losses[bit], dtype=torch.float64).flatten()\n              for bit in LEVELS}\n    lengths = {tensor.numel() for tensor in values.values()}\n    if len(lengths) != 1 or next(iter(lengths), 0) == 0:\n        raise ValueError("all loss vectors must have the same non-zero length")\n    if any(not torch.isfinite(tensor).all() for tensor in values.values()):\n        raise ValueError("loss vectors must be finite")\n    out_features = next(iter(lengths))\n    counts = _rounded_counts(out_features, budget, alignment)\n\n    # Upgrades are ranked by the error removed by moving up one level.\n    benefit8 = values[4] - values[8]\n    benefit16 = values[8] - values[16]\n    # Choose the final classes hierarchically.  An FP16 channel is upgraded\n    # from INT8, so it must be selected before INT8 candidates are considered.\n    selected: Dict[int, list[int]] = {4: [], 8: [], 16: []}\n    selected[16] = [index for _, index in sorted(\n        ((float(benefit16[i]), i) for i in range(out_features)),\n        key=lambda item: (-item[0], item[1]))[:counts[16]]]\n    used = set(selected[16])\n    selected[8] = [index for _, index in sorted(\n        ((float(benefit8[i]), i) for i in range(out_features) if i not in used),\n        key=lambda item: (-item[0], item[1]))[:counts[8]]]\n    used.update(selected[8])\n    selected[4] = [index for index in range(out_features) if index not in used]\n    if len(selected[4]) != counts[4]:\n        raise ValueError("budget/alignment could not be represented exactly")\n    result = ThreeLevelAllocation(\n        indices={bit: tuple(sorted(selected[bit])) for bit in LEVELS},\n        scores={4: tuple(float(x) for x in values[4]),\n                8: tuple(float(x) for x in benefit8),\n                16: tuple(float(x) for x in benefit16)},\n        budget=budget,\n    )\n    result.verify(out_features)\n    return result\n\n\ndef allocate_model_channels(\n    layer_losses: Mapping[str, Mapping[int, torch.Tensor | Sequence[float]]],\n    budget: ThreeLevelBudget,\n    alignment: int = 1,\n    layer_minimums: Mapping[str, int] | None = None,\n    metadata: Dict[str, object] | None = None,\n) -> Dict[str, ThreeLevelAllocation]:\n    """Allocate one global bit budget across all output channels in all layers.\n\n    The returned per-layer allocations preserve original channel indices.  A\n    layer minimum is expressed as the minimum number of non-INT4 channels and\n    is satisfied before global marginal-benefit ranking.  This is the clean\n    replacement for the upstream searcher\'s per-iteration percentage split.\n    """\n    if not layer_losses:\n        raise ValueError("layer_losses must contain at least one layer")\n    minimums = dict(layer_minimums or {})\n    flattened = []\n    per_layer = {}\n    for name, losses in layer_losses.items():\n        values = {bit: torch.as_tensor(losses[bit], dtype=torch.float64).flatten()\n                  for bit in LEVELS}\n        lengths = {value.numel() for value in values.values()}\n        if len(lengths) != 1 or not lengths or next(iter(lengths)) == 0:\n            raise ValueError(f"invalid loss vectors for layer {name}")\n        n = next(iter(lengths))\n        if alignment > 1 and n < alignment:\n            raise ValueError(f"layer {name} is smaller than alignment")\n        if minimums.get(name, 0) < 0 or minimums.get(name, 0) > n:\n            raise ValueError(f"invalid layer minimum for {name}")\n        per_layer[name] = values\n        flattened.extend((float(values[4][i] - values[8][i]),\n                          float(values[8][i] - values[16][i]), name, i)\n                         for i in range(n))\n\n    total = len(flattened)\n    counts = _rounded_counts(total, budget, alignment)\n    required = sum(minimums.values())\n    if required > counts[8] + counts[16]:\n        raise ValueError("layer minimums exceed non-INT4 global budget")\n    chosen: Dict[str, Dict[int, set[int]]] = {\n        name: {4: set(), 8: set(), 16: set()} for name in per_layer\n    }\n    ranked16 = sorted(((benefit16, name, index)\n                       for _, benefit16, name, index in flattened), reverse=True)\n    for _, name, index in ranked16[:counts[16]]:\n        chosen[name][16].add(index)\n    used = {(name, index) for name in chosen for index in chosen[name][16]}\n    ranked8 = sorted(((benefit8, name, index)\n                      for benefit8, _, name, index in flattened\n                      if (name, index) not in used), reverse=True)\n    for _, name, index in ranked8[:counts[8]]:\n        chosen[name][8].add(index)\n    used.update((name, index) for name in chosen for index in chosen[name][8])\n\n    # Meet per-layer minimums without changing global precision counts.  Each\n    # swap replaces the weakest selected channel from a donor layer that stays\n    # above its own minimum with the strongest candidate from the deficient one.\n    def selected_count(layer: str) -> int:\n        return len(chosen[layer][8]) + len(chosen[layer][16])\n\n    for name in sorted(per_layer):\n        while selected_count(name) < minimums.get(name, 0):\n            candidates = []\n            for index in range(per_layer[name][4].numel()):\n                if (name, index) in used:\n                    continue\n                candidates.extend((\n                    (float(per_layer[name][4][index] - per_layer[name][8][index]),\n                     8, index),\n                    (float(per_layer[name][8][index] - per_layer[name][16][index]),\n                     16, index),\n                ))\n            swapped = False\n            for _, bit, incoming in sorted(candidates, reverse=True):\n                donors = []\n                for donor in per_layer:\n                    if selected_count(donor) <= minimums.get(donor, 0):\n                        continue\n                    for outgoing in chosen[donor][bit]:\n                        values = per_layer[donor]\n                        benefit = (values[4][outgoing] - values[8][outgoing]\n                                   if bit == 8 else\n                                   values[8][outgoing] - values[16][outgoing])\n                        donors.append((float(benefit), donor, outgoing))\n                if not donors:\n                    continue\n                _, donor, outgoing = min(donors)\n                chosen[donor][bit].remove(outgoing)\n                chosen[name][bit].add(incoming)\n                used.remove((donor, outgoing))\n                used.add((name, incoming))\n                swapped = True\n                break\n            if not swapped:\n                raise ValueError("layer minimums cannot be satisfied with exact budgets")\n    result = {}\n    for name, values in per_layer.items():\n        indices = {bit: tuple(sorted(chosen[name][bit])) for bit in LEVELS}\n        indices[4] = tuple(i for i in range(values[4].numel())\n                           if all(i not in chosen[name][bit] for bit in (8, 16)))\n        allocation = ThreeLevelAllocation(\n            indices=indices,\n            scores={4: tuple(float(x) for x in values[4]),\n                    8: tuple(float(x) for x in values[4] - values[8]),\n                    16: tuple(float(x) for x in values[8] - values[16])},\n            budget=budget,\n            metadata={**(metadata or {}), "layer_name": name,\n                      "global_channel_count": total},\n        )\n        allocation.verify(values[4].numel())\n        result[name] = allocation\n    return result\n\n\n@torch.no_grad()\ndef estimate_channel_losses(activation: torch.Tensor, weight: torch.Tensor,\n                            group_size: int = 128\n                            ) -> tuple[Dict[int, torch.Tensor], torch.Tensor]:\n    """Estimate activation-aware output MSE for all three precision choices.\n\n    The same activation subset and group rules are used for every precision.\n    The returned AWQ statistic is kept separate and can be used as a tie-breaker\n    in an ablation without silently changing the marginal-loss objective.\n    """\n    if activation.shape[-1] != weight.shape[1] or weight.dim() != 2:\n        raise ValueError("activation and weight shapes are incompatible")\n    if weight.shape[1] % group_size:\n        raise ValueError("in_features must be divisible by group_size")\n    x = activation.reshape(-1, activation.shape[-1]).float()\n    reference = torch.nn.functional.linear(x, weight.float())\n    losses = {}\n    all_fp16 = ThreeLevelAllocation(\n        indices={4: (), 8: (), 16: tuple(range(weight.shape[0]))},\n        scores={bit: (0.0,) * weight.shape[0] for bit in LEVELS},\n        budget=ThreeLevelBudget(0, 0, 100),\n    )\n    for bit in LEVELS:\n        indices = {level: () for level in LEVELS}\n        indices[bit] = tuple(range(weight.shape[0]))\n        allocation = ThreeLevelAllocation(\n            indices=indices, scores=all_fp16.scores,\n            budget=ThreeLevelBudget(100 if bit == 4 else 0,\n                                    100 if bit == 8 else 0,\n                                    100 if bit == 16 else 0),\n        )\n        output = fake_linear(x, weight.float(), allocation, group_size)\n        losses[bit] = (output - reference).pow(2).mean(dim=0).cpu()\n    awq_stat = x.abs().mean(dim=0).cpu()\n    return losses, awq_stat\n\n\ndef fake_linear(x: torch.Tensor, weight: torch.Tensor,\n                allocation: ThreeLevelAllocation,\n                group_size: int = 128) -> torch.Tensor:\n    """Reference output for a three-level weight-only linear layer."""\n    if weight.dim() != 2 or x.shape[-1] != weight.shape[1]:\n        raise ValueError("incompatible x/weight shapes")\n    output = torch.empty((*x.shape[:-1], weight.shape[0]), dtype=x.dtype, device=x.device)\n    for bit in LEVELS:\n        indices = allocation.indices[bit]\n        if not indices:\n            continue\n        chunk = weight[list(indices)]\n        if bit == 16:\n            quantized = chunk\n        else:\n            if chunk.shape[1] % group_size:\n                raise ValueError("in_features must be divisible by group_size")\n            groups = chunk.reshape(chunk.shape[0], -1, group_size)\n            if bit == 8:\n                maximum = groups.abs().amax(dim=-1, keepdim=True).clamp_min(1e-5)\n                quantized = (groups / (maximum / 127)).round().clamp(-128, 127) * (maximum / 127)\n            else:\n                minimum = groups.amin(dim=-1, keepdim=True)\n                maximum = groups.amax(dim=-1, keepdim=True)\n                scale = (maximum - minimum).clamp_min(1e-5) / 15\n                zero = (-minimum / scale).round().clamp(0, 15)\n                quantized = ((groups / scale).round() + zero).clamp(0, 15)\n                quantized = (quantized - zero) * scale\n            quantized = quantized.reshape_as(chunk)\n        output[..., list(indices)] = torch.nn.functional.linear(x, quantized)\n    return output', 'mixllm/nn/modules/mixllm_config.py': '# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n\nimport os\nimport json\nfrom typing import Dict, Optional, List, Tuple\nfrom dataclasses import dataclass, field\nfrom transformers.utils.hub import PushToHubMixin\n\n\n@dataclass\nclass MixLLMConfig(PushToHubMixin):\n    quant_method: str = field(default="mixllm")\n    ratio: float = field(default=1.0)\n    # Legacy MixLLM uses ratio as the INT8 fraction.  These fields are\n    # optional so existing two-level checkpoints remain loadable.\n    precision_percentages: Optional[Dict[str, int]] = None\n    allocation_file: Optional[str] = None\n    allocation_version: int = 1\n    group_size: int = 128\n    backend: str = "auto"\n    format_version: int = 2\n    model_revision: Optional[str] = None\n    calibration_seed: Optional[int] = None\n    dtype: str = "float16"\n    kernel_capability: Optional[Tuple[int, int]] = None\n    config_file_name = "config.json"\n    modules_to_not_convert: Optional[List] = None\n\n    @classmethod\n    def from_dict(cls, quant_config: Dict = {}):\n        if not quant_config:\n            quant_config = cls()\n        else:\n            supported = {\n                key: value for key, value in quant_config.items()\n                if key in cls.__dataclass_fields__\n            }\n            quant_config = cls(**supported)\n\n        if quant_config.precision_percentages is None and quant_config.ratio != 1.0:\n            int8 = round(float(quant_config.ratio) * 100)\n            quant_config.precision_percentages = {"4": 100 - int8, "8": int8, "16": 0}\n        if quant_config.precision_percentages is not None:\n            percentages = quant_config.precision_percentages\n            normalized = {str(bit): int(percentages.get(str(bit), 0))\n                          for bit in (4, 8, 16)}\n            if any(value < 0 for value in normalized.values()) or sum(normalized.values()) != 100:\n                raise ValueError("precision_percentages for 4/8/16 must sum to 100")\n            quant_config.precision_percentages = normalized\n        if quant_config.group_size <= 0:\n            raise ValueError("group_size must be positive")\n        if quant_config.format_version not in (1, 2):\n            raise ValueError("unsupported MixLLM config format_version")\n        if quant_config.dtype not in {"float16", "bfloat16"}:\n            raise ValueError("dtype must be float16 or bfloat16")\n        if quant_config.kernel_capability is not None:\n            capability = tuple(int(value) for value in quant_config.kernel_capability)\n            if len(capability) != 2 or any(value < 0 for value in capability):\n                raise ValueError("kernel_capability must be a CUDA (major, minor) pair")\n            quant_config.kernel_capability = capability\n\n        return quant_config\n\n    def to_dict(self):\n        return {\n            "ratio": self.ratio,\n            "modules_to_not_convert": self.modules_to_not_convert,\n            "precision_percentages": self.precision_percentages,\n            "allocation_file": self.allocation_file,\n            "allocation_version": self.allocation_version,\n            "group_size": self.group_size,\n            "backend": self.backend,\n            "format_version": self.format_version,\n            "model_revision": self.model_revision,\n            "calibration_seed": self.calibration_seed,\n            "dtype": self.dtype,\n            "kernel_capability": self.kernel_capability,\n        }\n\n    def to_transformers_dict(self):\n        return {\n            "quant_method": self.quant_method,\n            "ratio": self.ratio,\n            "modules_to_not_convert": self.modules_to_not_convert,\n            "precision_percentages": self.precision_percentages,\n            "allocation_file": self.allocation_file,\n            "allocation_version": self.allocation_version,\n            "group_size": self.group_size,\n            "backend": self.backend,\n            "format_version": self.format_version,\n            "model_revision": self.model_revision,\n            "calibration_seed": self.calibration_seed,\n            "dtype": self.dtype,\n            "kernel_capability": self.kernel_capability,\n        }\n\n    def from_transformers_dict(self, transformers_dict: Dict):\n        return {\n            "quant_method":\n                transformers_dict.get("quant_method"),\n            "ratio":\n                transformers_dict.get("ratio"),\n            "modules_to_not_convert":\n                transformers_dict.get("modules_to_not_convert"),\n            "precision_percentages":\n                transformers_dict.get("precision_percentages"),\n            "allocation_file": transformers_dict.get("allocation_file"),\n            "allocation_version": transformers_dict.get("allocation_version", 1),\n            "group_size": transformers_dict.get("group_size", 128),\n            "backend": transformers_dict.get("backend", "auto"),\n            "format_version": transformers_dict.get("format_version", 1),\n            "model_revision": transformers_dict.get("model_revision"),\n            "calibration_seed": transformers_dict.get("calibration_seed"),\n            "dtype": transformers_dict.get("dtype", "float16"),\n            "kernel_capability": transformers_dict.get("kernel_capability"),\n        }\n', 'mixllm/nn/modules/three_level_linear.py': '"""Packed FP16/INT8/INT4 output-feature linear for MixLLM."""\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\nimport torch\nfrom torch import nn\n\nfrom mixllm.quantization.three_level import LEVELS, ThreeLevelAllocation\n\n\ndef _pack_uint4(codes: torch.Tensor) -> torch.Tensor:\n    if codes.shape[-1] % 2:\n        raise ValueError("INT4 input width must be even")\n    values = codes.to(torch.uint8)\n    return (values[..., 0::2] | (values[..., 1::2] << 4)).contiguous()\n\n\ndef _unpack_uint4(packed: torch.Tensor) -> torch.Tensor:\n    output = torch.empty((*packed.shape[:-1], packed.shape[-1] * 2),\n                         dtype=torch.uint8, device=packed.device)\n    output[..., 0::2] = packed & 0x0F\n    output[..., 1::2] = packed >> 4\n    return output\n\n\nclass ThreeLevelLinear(nn.Module):\n    """Reference backend and serialization contract for a three-level op."""\n\n    quant_method = "mixllm_three_level"\n\n    def __init__(self, in_features: int, out_features: int, group_size: int = 128,\n                 bias: Optional[torch.Tensor] = None) -> None:\n        super().__init__()\n        if in_features <= 0 or out_features <= 0 or group_size <= 0:\n            raise ValueError("linear dimensions and group_size must be positive")\n        if in_features % group_size:\n            raise ValueError("in_features must be divisible by group_size")\n        self.in_features = in_features\n        self.out_features = out_features\n        self.group_size = group_size\n        self.register_buffer("weight_fp16", torch.empty(0, in_features, dtype=torch.float16))\n        self.register_buffer("weight_int8", torch.empty(0, in_features, dtype=torch.int8))\n        self.register_buffer("scale_int8", torch.empty(0, in_features // group_size, dtype=torch.float16))\n        self.register_buffer("weight_int4", torch.empty(0, in_features // 2, dtype=torch.uint8))\n        self.register_buffer("scale_int4", torch.empty(0, in_features // group_size, dtype=torch.float16))\n        self.register_buffer("zero_int4", torch.empty(0, in_features // group_size, dtype=torch.uint8))\n        for bit in LEVELS:\n            self.register_buffer(f"indices_{bit}", torch.empty(0, dtype=torch.int32))\n        self.register_buffer("bias", None if bias is None else bias.detach().to(torch.float16))\n\n    @classmethod\n    @torch.no_grad()\n    def from_weight(cls, weight: torch.Tensor, allocation: ThreeLevelAllocation,\n                    group_size: int = 128,\n                    bias: Optional[torch.Tensor] = None) -> "ThreeLevelLinear":\n        if weight.dim() != 2:\n            raise ValueError("weight must have shape [out_features, in_features]")\n        out_features, in_features = weight.shape\n        allocation.verify(out_features)\n        layer = cls(in_features, out_features, group_size, bias).to(weight.device)\n        groups = in_features // group_size\n\n        for bit in LEVELS:\n            indices = torch.tensor(allocation.indices[bit], dtype=torch.int32,\n                                   device=weight.device)\n            setattr(layer, f"indices_{bit}", indices)\n            if not indices.numel():\n                continue\n            chunk = weight.index_select(0, indices.long()).float()\n            if bit == 16:\n                layer.weight_fp16 = chunk.to(torch.float16).contiguous()\n            elif bit == 8:\n                viewed = chunk.reshape(-1, groups, group_size)\n                scale = (viewed.abs().amax(-1) / 127).clamp_min(1e-5)\n                codes = (viewed / scale.unsqueeze(-1)).round().clamp(-128, 127)\n                layer.weight_int8 = codes.to(torch.int8).reshape(-1, in_features).contiguous()\n                layer.scale_int8 = scale.to(torch.float16).contiguous()\n            else:\n                viewed = chunk.reshape(-1, groups, group_size)\n                minimum, maximum = viewed.amin(-1), viewed.amax(-1)\n                scale = ((maximum - minimum) / 15).clamp_min(1e-5)\n                zero = (-minimum / scale).round().clamp(0, 15)\n                codes = ((viewed / scale.unsqueeze(-1)).round() + zero.unsqueeze(-1)).clamp(0, 15)\n                layer.weight_int4 = _pack_uint4(codes.to(torch.uint8).reshape(-1, in_features))\n                layer.scale_int4 = scale.to(torch.float16).contiguous()\n                layer.zero_int4 = zero.to(torch.uint8).contiguous()\n        return layer\n\n    @torch.no_grad()\n    def dequantize_weight(self) -> torch.Tensor:\n        device = self.weight_fp16.device\n        output = torch.empty(self.out_features, self.in_features,\n                             dtype=torch.float32, device=device)\n        if self.indices_16.numel():\n            output.index_copy_(0, self.indices_16.long(), self.weight_fp16.float())\n        if self.indices_8.numel():\n            scales = self.scale_int8.float().repeat_interleave(self.group_size, dim=1)\n            output.index_copy_(0, self.indices_8.long(), self.weight_int8.float() * scales)\n        if self.indices_4.numel():\n            codes = _unpack_uint4(self.weight_int4).float()\n            scales = self.scale_int4.float().repeat_interleave(self.group_size, dim=1)\n            zeros = self.zero_int4.float().repeat_interleave(self.group_size, dim=1)\n            output.index_copy_(0, self.indices_4.long(), (codes - zeros) * scales)\n        return output\n\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\n                              missing_keys, unexpected_keys, error_msgs):\n        # Two-level checkpoints predate the FP16 partition. Treat its omitted\n        # tensors as an empty partition while preserving strict loading for all\n        # other packed state.\n        legacy_defaults = {\n            "weight_fp16": self.weight_fp16,\n            "indices_16": self.indices_16,\n        }\n        for name, value in legacy_defaults.items():\n            state_dict.setdefault(prefix + name, value)\n        variable_buffers = (\n            "weight_fp16", "weight_int8", "scale_int8", "weight_int4",\n            "scale_int4", "zero_int4", "indices_4", "indices_8", "indices_16",\n        )\n        for name in variable_buffers:\n            key = prefix + name\n            if key in state_dict:\n                setattr(self, name, torch.empty_like(state_dict[key], device=self.weight_fp16.device))\n        super()._load_from_state_dict(state_dict, prefix, local_metadata, strict,\n                                      missing_keys, unexpected_keys, error_msgs)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.shape[-1] != self.in_features:\n            raise ValueError(f"expected input width {self.in_features}, got {x.shape[-1]}")\n        result = torch.nn.functional.linear(x.float(), self.dequantize_weight(),\n                                            None if self.bias is None else self.bias.float())\n        return result.to(x.dtype)', 'mixllm/nn/modules/ops.py': '# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n\nimport torch\n\n__all__ = ["quantize", "transpose", "mixllm_gemm", "mixllm_three_level_gemm"]\n\n\ndef transpose(a):\n    return torch.ops.kernels_mixllm.transpose(a)\n\n\ndef quantize(a):\n    return torch.ops.kernels_mixllm.quantize(a)\n\n\ndef mixllm_gemm(a, scale_act, zero, scale_int8, scale_int4, indices_int8,\n                indices_int4, b_int8, b_int4):\n    return torch.ops.kernels_mixllm.gemm(a, scale_act, zero, scale_int8,\n                                         scale_int4, indices_int8, indices_int4,\n                                         b_int8, b_int4)\n\n\ndef mixllm_three_level_gemm(a, scale_act, zero_int4, scale_int8,\n                            scale_int4, indices_int8, indices_int4,\n                            indices_fp16, b_int8, b_int4, b_fp16):\n    """Correctness-first 4/8/16 ABI using the existing MixLLM quantized op.\n\n    The quantized partitions run through the upstream CUDA extension. The FP16\n    partition is computed with PyTorch GEMM, and all outputs are scattered to\n    their original output-feature positions. A future fused op can replace this\n    implementation without changing the checkpoint contract.\n    """\n    partitions = (indices_int4, indices_int8, indices_fp16)\n    total_n = sum(indices.numel() for indices in partitions)\n    if total_n == 0:\n        raise ValueError("at least one output partition is required")\n    complete = torch.cat(partitions).long()\n    if complete.unique().numel() != total_n or complete.min().item() != 0 or complete.max().item() != total_n - 1:\n        raise ValueError("4/8/16 indices must form a complete output-channel partition")\n\n    m, k = a.shape\n    if k % 128:\n        raise ValueError("current MixLLM CUDA ABI requires K divisible by 128")\n    output = torch.empty((m, total_n), dtype=torch.float16, device=a.device)\n    n8, n4 = indices_int8.numel(), indices_int4.numel()\n    if n8 + n4:\n        local_int8 = torch.arange(n8, dtype=torch.int32, device=a.device)\n        local_int4 = torch.arange(n8, n8 + n4, dtype=torch.int32, device=a.device)\n        quantized = mixllm_gemm(\n            a, scale_act, zero_int4, scale_int8, scale_int4,\n            local_int8, local_int4, b_int8, b_int4,\n        )\n        if n8 and n4:\n            quantized = quantized.t().contiguous()\n        output.index_copy_(1, torch.cat((indices_int8, indices_int4)).long(), quantized)\n\n    if indices_fp16.numel():\n        groups = k // 128\n        activation = (\n            a.float().reshape(m, groups, 128)\n            * scale_act[:, :m].t().reshape(m, groups, 1).float()\n        ).reshape(m, k)\n        fp16_result = activation @ b_fp16.float().t()\n        output.index_copy_(1, indices_fp16.long(), fp16_result.to(torch.float16))\n    return output\n\n\n@torch.library.register_fake("kernels_mixllm::quantize")\ndef quantize_abstract(a):\n    torch._check(a.dim() == 2, "Input must be a 2D tensor")\n    m = a.shape[0]\n    n = a.shape[1]\n    group_size = 128\n    torch._check(a.is_cuda, "Input must be on CUDA device")\n    torch._check(a.dtype == torch.float16, "Input must be float16")\n    torch._check(\n        n % group_size == 0,\n        "Input must have a second dimension that is a multiple of group_size")\n\n    m_round_even = m + (m % 2)\n    return (torch.empty((m, n), dtype=torch.int8, device="cuda:0"),\n            torch.empty((n // group_size, m_round_even),\n                        dtype=torch.float16,\n                        device="cuda:0"))\n\n\n@torch.library.register_fake("kernels_mixllm::transpose")\ndef transpose_abstract(a):\n    torch._check(a.dim() == 2, "Input must be a 2D tensor")\n    m = a.shape[0]\n    n = a.shape[1]\n    torch._check(a.is_cuda, "Input must be on CUDA device")\n    torch._check(a.dtype == torch.float16, "Input must be float16")\n    return torch.empty((n, m), dtype=torch.float16, device="cuda:0")\n\n\n@torch.library.register_fake("kernels_mixllm::gemm")\ndef mixllm_gemm_abstract(a, scale_act, zero, scale_int8, scale_int4,\n                         indices_int8, indices_int4, b_int8, b_int4):\n    torch._check(a.is_cuda, "Input tensor A must be on CUDA device")\n    torch._check(a.dtype == torch.int8, "Input tensor A must be int8")\n    torch._check(scale_act.dtype == torch.float16,\n                 "Scale activation tensor must be float16")\n    torch._check(zero.dtype == torch.uint8, "Zero tensor must be uint8")\n    torch._check(scale_int8.dtype == torch.float16,\n                 "Scale int8 tensor must be float16")\n    torch._check(scale_int4.dtype == torch.float16,\n                 "Scale int4 tensor must be float16")\n    torch._check(indices_int8.dtype == torch.int32,\n                 "Indices int8 tensor must be int32")\n    torch._check(indices_int4.dtype == torch.int32,\n                 "Indices int4 tensor must be int32")\n    torch._check(b_int8.dtype == torch.int8, "B int8 tensor must be int8")\n    torch._check(b_int4.dtype == torch.uint8, "B int4 tensor must be uint8")\n    torch._check(b_int8.is_cuda, "B int8 tensor must be on CUDA device")\n    torch._check(b_int4.is_cuda, "B int4 tensor must be on CUDA device")\n    torch._check(a.dim() == 2, "Input tensor A must be 2D")\n    torch._check(scale_act.dim() == 2, "Scale activation tensor must be 2D")\n    torch._check(zero.dim() == 2, "Zero tensor must be 2D")\n    torch._check(scale_int8.dim() == 2, "Scale int8 tensor must be 2D")\n    torch._check(scale_int4.dim() == 2, "Scale int4 tensor must be 2D")\n    torch._check(indices_int8.dim() == 1, "Indices int8 tensor must be 1D")\n    torch._check(indices_int4.dim() == 1, "Indices int4 tensor must be 1D")\n    torch._check(b_int8.dim() == 2, "B int8 tensor must be 2D")\n    torch._check(b_int4.dim() == 2, "B int4 tensor must be 2D")\n\n    m = a.shape[0]\n    n = (indices_int4.numel() + indices_int8.numel())\n    is_row_major = (indices_int4.numel() == 0 or indices_int8.numel() == 0)\n    if is_row_major:\n        c = torch.empty((m, n), dtype=torch.float16, device="cuda:0")\n    else:\n        c = torch.empty((n, m), dtype=torch.float16, device="cuda:0")\n\n    return c\n', 'mixllm/runtime_capability.py': '"""Runtime capability gates for the MixLLM CUDA backends."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\n\n@dataclass(frozen=True)\nclass RuntimeCapability:\n    major: int\n    minor: int\n    requested_backend: str = "auto"\n    sm75_available: bool = False\n\n    @property\n    def compute_capability(self) -> float:\n        return self.major + self.minor / 10\n\n    @property\n    def supports_ampere_mixllm(self) -> bool:\n        return self.major >= 8\n\n    @property\n    def supports_sm75_backend(self) -> bool:\n        return self.sm75_available and (self.major, self.minor) == (7, 5)\n\n    def select_backend(self) -> str:\n        if self.requested_backend not in {"auto", "sm75", "ampere", "reference"}:\n            raise ValueError(f"unknown MixLLM backend: {self.requested_backend}")\n        if self.requested_backend == "reference":\n            return "reference"\n        if self.requested_backend == "ampere":\n            if not self.supports_ampere_mixllm:\n                raise RuntimeError("Ampere MixLLM backend requires compute capability >= 8.0")\n            return "ampere"\n        if self.requested_backend == "sm75":\n            if not self.supports_sm75_backend:\n                raise RuntimeError(\n                    "SM75 backend requires compute capability 7.5 and a validated SM75 build"\n                )\n            return "sm75"\n        if self.supports_ampere_mixllm:\n            return "ampere"\n        if self.supports_sm75_backend:\n            return "sm75"\n        return "reference"\n\n\ndef detect_runtime(torch_module) -> Optional[RuntimeCapability]:\n    """Detect CUDA capability without importing torch at module import time."""\n    if not torch_module.cuda.is_available():\n        return None\n    major, minor = torch_module.cuda.get_device_capability()\n    return RuntimeCapability(major, minor)', 'mixllm/sm75_backend.py': '"""Build and invoke the standalone SM75 three-level correctness backend."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Dict, Iterable, Optional\n\n\n_LOADED = False\n\n\ndef load_sm75_backend(torch_module, build_directory: Optional[str | Path] = None) -> None:\n    """JIT-build the SM75-only torch operator and load it into this process."""\n    global _LOADED\n    if _LOADED:\n        return\n    if not torch_module.cuda.is_available():\n        raise RuntimeError("SM75 backend requires CUDA")\n    capability = tuple(torch_module.cuda.get_device_capability())\n    if capability != (7, 5):\n        raise RuntimeError(f"SM75 backend requires capability 7.5, got {capability}")\n\n    from torch.utils.cpp_extension import load\n\n    source = Path(__file__).resolve().parent / "kernels" / "three_level_sm75.cu"\n    kwargs = {}\n    if build_directory is not None:\n        directory = Path(build_directory)\n        directory.mkdir(parents=True, exist_ok=True)\n        kwargs["build_directory"] = str(directory)\n    load(\n        name="mixllm_sm75_backend",\n        sources=[str(source)],\n        extra_cuda_cflags=["-O3", "-lineinfo", "-gencode=arch=compute_75,code=sm_75"],\n        extra_cflags=["-O3"],\n        is_python_module=False,\n        verbose=True,\n        **kwargs,\n    )\n    _LOADED = True\n\n\ndef three_level_linear(module, x, torch_module):\n    """Run a packed ``ThreeLevelLinear`` through the loaded SM75 operator."""\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before using the SM75 operator")\n    if x.dim() != 2:\n        raise ValueError("SM75 correctness backend currently requires a 2D input")\n    indices = (module.indices_4, module.indices_8, module.indices_16)\n    # CUDA Graph capture forbids validation kernels and host synchronisation on\n    # the capture stream. Validate eagerly on ordinary calls; packed modules\n    # produced by ThreeLevelLinear.from_weight have already passed this check.\n    is_capturing = bool(torch_module.cuda.is_current_stream_capturing())\n    if not is_capturing:\n        complete = torch_module.cat(indices).long()\n        expected = torch_module.arange(module.out_features, device=x.device)\n        if complete.numel() != module.out_features or not torch_module.equal(\n                complete.sort().values, expected):\n            raise ValueError("4/8/16 indices must form a complete output partition")\n    tensors = (\n        module.weight_int4, module.scale_int4, module.zero_int4, module.indices_4,\n        module.weight_int8, module.scale_int8, module.indices_8,\n        module.weight_fp16, module.indices_16,\n    )\n    if any(tensor.device != x.device for tensor in tensors):\n        raise ValueError("input and all packed tensors must be on the same device")\n    return torch_module.ops.mixllm_sm75.three_level_linear(\n        x.contiguous(), *(tensor.contiguous() for tensor in tensors), module.group_size,\n    )\n\n\ndef benchmark_sm75_backend(\n    module,\n    rows: Iterable[int],\n    torch_module,\n    warmup: int = 10,\n    iterations: int = 50,\n) -> Dict[str, object]:\n    """Measure the SM75 operator against its dequantized dense reference.\n\n    Timings use per-iteration CUDA events and synchronize only after all events\n    have been recorded. This is a kernel-level benchmark, not tokens/second.\n    """\n    if warmup < 1 or iterations < 2:\n        raise ValueError("benchmark requires warmup >= 1 and iterations >= 2")\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before benchmarking")\n    dense_weight = module.dequantize_weight()\n    results = []\n\n    def measure(callable_):\n        for _ in range(warmup):\n            callable_()\n        torch_module.cuda.synchronize()\n        pairs = []\n        for _ in range(iterations):\n            start = torch_module.cuda.Event(enable_timing=True)\n            end = torch_module.cuda.Event(enable_timing=True)\n            start.record()\n            callable_()\n            end.record()\n            pairs.append((start, end))\n        torch_module.cuda.synchronize()\n        values = sorted(float(start.elapsed_time(end)) for start, end in pairs)\n        p95_index = min(len(values) - 1, int(0.95 * len(values)))\n        return {"p50_ms": median(values), "p95_ms": values[p95_index]}\n\n    for row_count in rows:\n        if int(row_count) <= 0:\n            raise ValueError("benchmark row counts must be positive")\n        x = torch_module.randn(\n            int(row_count), module.in_features, device=module.weight_fp16.device,\n            dtype=torch_module.float16,\n        )\n        actual = three_level_linear(module, x, torch_module)\n        expected = torch_module.nn.functional.linear(x.float(), dense_weight)\n        max_error = float((actual - expected).abs().max().item())\n        sm75 = measure(lambda: three_level_linear(module, x, torch_module))\n        dense = measure(lambda: torch_module.nn.functional.linear(x.float(), dense_weight))\n        results.append({\n            "rows": int(row_count),\n            "sm75": sm75,\n            "dense_fp32_reference": dense,\n            "p50_ratio_vs_dense": sm75["p50_ms"] / max(dense["p50_ms"], 1e-9),\n            "max_abs_error": max_error,\n        })\n    return {"status": "measured", "shapes": results}', 'mixllm/vllm_three_level.py': '"""Version-neutral helpers used by the pinned vLLM three-level patch."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Iterable, Tuple\n\nfrom mixllm.runtime_capability import RuntimeCapability\n\n\nPINNED_VLLM_VERSION = "0.9.0"\nPINNED_VLLM_COMMIT = "5fbbfe9a4c13094ad72ed3d6b4ef208a7ddc0fd7"\n\n\n@dataclass(frozen=True)\nclass PrecisionPercentages:\n    bit4: int\n    bit8: int\n    bit16: int\n\n    def __post_init__(self) -> None:\n        values = (self.bit4, self.bit8, self.bit16)\n        if any(value < 0 for value in values) or sum(values) != 100:\n            raise ValueError("4/8/16 percentages must sum to 100")\n\n\n@dataclass(frozen=True)\nclass VLLMThreeLevelConfig:\n    group_size: int\n    percentages: PrecisionPercentages\n    backend: str = "auto"\n    allocation_version: int = 1\n\n    @classmethod\n    def from_quantization_config(cls, config: Dict) -> "VLLMThreeLevelConfig":\n        if config.get("quant_method") not in {"mixllm_three_level", "mixllm"}:\n            raise ValueError("checkpoint is not a MixLLM three-level checkpoint")\n        raw = config.get("precision_percentages")\n        if raw is None:\n            raise ValueError("precision_percentages is required for three-level inference")\n        budget = PrecisionPercentages(\n            int(raw.get("4", raw.get(4, 0))),\n            int(raw.get("8", raw.get(8, 0))),\n            int(raw.get("16", raw.get(16, 0))),\n        )\n        group_size = int(config.get("group_size", 128))\n        if group_size != 128:\n            raise ValueError("current MixLLM CUDA ABI requires group_size=128")\n        return cls(\n            group_size=group_size,\n            percentages=budget,\n            backend=str(config.get("backend", "auto")),\n            allocation_version=int(config.get("allocation_version", 1)),\n        )\n\n    def select_backend(self, capability: Tuple[int, int],\n                       sm75_available: bool = False) -> str:\n        return RuntimeCapability(*capability, self.backend, sm75_available).select_backend()\n\n\ndef validate_partition_indices(indices: Dict[int, Iterable[int]], output_size: int) -> None:\n    values = [int(index) for bit in (4, 8, 16) for index in indices.get(bit, ())]\n    if sorted(values) != list(range(output_size)):\n        raise ValueError("vLLM partition indices must cover every output channel exactly once")\n\n\ndef remap_partition_indices_for_tp(\n    indices: Dict[int, Iterable[int]],\n    global_output_size: int,\n    shard_start: int,\n    shard_size: int,\n) -> Dict[int, Tuple[int, ...]]:\n    """Map checkpoint-global output indices to one tensor-parallel shard.\n\n    vLLM loads output-parallel weights as contiguous ranges. The packed tensors\n    retain their precision order, while the operator receives indices local to\n    the current shard so its output remains in original local-channel order.\n    """\n    validate_partition_indices(indices, global_output_size)\n    if shard_start < 0 or shard_size <= 0 or shard_start + shard_size > global_output_size:\n        raise ValueError("invalid tensor-parallel output shard")\n    shard_end = shard_start + shard_size\n    local = {\n        bit: tuple(int(index) - shard_start for index in indices.get(bit, ())\n                   if shard_start <= int(index) < shard_end)\n        for bit in (4, 8, 16)\n    }\n    validate_partition_indices(local, shard_size)\n    return local\n\n\ndef restore_global_partition_indices(\n    local_indices: Dict[int, Iterable[int]], shard_start: int, shard_size: int,\n) -> Dict[int, Tuple[int, ...]]:\n    """Restore checkpoint-global indices after a local shard round trip."""\n    validate_partition_indices(local_indices, shard_size)\n    if shard_start < 0:\n        raise ValueError("shard_start must be non-negative")\n    return {\n        bit: tuple(int(index) + shard_start for index in local_indices.get(bit, ()))\n        for bit in (4, 8, 16)\n    }', 'mixllm/kernels/three_level_sm75.cu': '// Copyright (c) Microsoft Corporation.\n// SPDX-License-Identifier: MIT\n\n#include <ATen/cuda/CUDAContext.h>\n#include <ATen/Dispatch.h>\n#include <ATen/Functions.h>\n#include <ATen/Tensor.h>\n#include <c10/cuda/CUDAGuard.h>\n#include <c10/cuda/CUDAException.h>\n#include <torch/library.h>\n\n#include <cuda.h>\n#include <cuda_fp16.h>\n#include <cuda_runtime.h>\n\nnamespace {\n\nconstexpr int kThreads = 128;\n\ntemplate <typename scalar_t>\n__device__ __forceinline__ float load_activation(const scalar_t* input,\n                                                  int64_t offset) {\n  return static_cast<float>(input[offset]);\n}\n\ntemplate <typename scalar_t>\n__global__ void three_level_partition_kernel(\n    const scalar_t* input, const uint8_t* packed_int4,\n    const __half* scale_int4, const uint8_t* zero_int4,\n    const int32_t* indices_int4, const int8_t* weight_int8,\n    const __half* scale_int8, const int32_t* indices_int8,\n    const __half* weight_fp16, const int32_t* indices_fp16, float* output,\n    int64_t rows, int64_t width, int64_t output_width, int64_t n4,\n    int64_t n8, int64_t n16, int64_t group_size) {\n  const int64_t row = blockIdx.y;\n  const int64_t partition_channel = blockIdx.x;\n  const int64_t groups = width / group_size;\n  const int64_t packed_width = width / 2;\n  float accumulator = 0.0f;\n\n  if (partition_channel < n4) {\n    const int64_t channel = partition_channel;\n    for (int64_t k = threadIdx.x; k < width; k += blockDim.x) {\n      const uint8_t packed = packed_int4[channel * packed_width + k / 2];\n      const uint8_t code = (k & 1) ? (packed >> 4) : (packed & 0x0f);\n      const int64_t group = channel * groups + k / group_size;\n      const float dequantized =\n          (static_cast<float>(code) - static_cast<float>(zero_int4[group])) *\n          __half2float(scale_int4[group]);\n      accumulator += load_activation(input, row * width + k) * dequantized;\n    }\n  } else if (partition_channel < n4 + n8) {\n    const int64_t channel = partition_channel - n4;\n    for (int64_t k = threadIdx.x; k < width; k += blockDim.x) {\n      const float dequantized =\n          static_cast<float>(weight_int8[channel * width + k]) *\n          __half2float(scale_int8[channel * groups + k / group_size]);\n      accumulator += load_activation(input, row * width + k) * dequantized;\n    }\n  } else {\n    const int64_t channel = partition_channel - n4 - n8;\n    for (int64_t k = threadIdx.x; k < width; k += blockDim.x) {\n      accumulator += load_activation(input, row * width + k) *\n                     __half2float(weight_fp16[channel * width + k]);\n    }\n  }\n\n  __shared__ float reduction[kThreads];\n  reduction[threadIdx.x] = accumulator;\n  __syncthreads();\n  for (int offset = kThreads / 2; offset > 32; offset >>= 1) {\n    if (threadIdx.x < offset) {\n      reduction[threadIdx.x] += reduction[threadIdx.x + offset];\n    }\n    __syncthreads();\n  }\n  if (threadIdx.x < 32) {\n    float value = reduction[threadIdx.x] + reduction[threadIdx.x + 32];\n#pragma unroll\n    for (int offset = 16; offset > 0; offset >>= 1) {\n      value += __shfl_down_sync(0xffffffff, value, offset);\n    }\n    if (threadIdx.x == 0) {\n      int32_t output_channel;\n      if (partition_channel < n4) {\n        output_channel = indices_int4[partition_channel];\n      } else if (partition_channel < n4 + n8) {\n        output_channel = indices_int8[partition_channel - n4];\n      } else {\n        output_channel = indices_fp16[partition_channel - n4 - n8];\n      }\n      output[row * output_width + output_channel] = value;\n    }\n  }\n}\n\nvoid check_cuda_contiguous(const at::Tensor& tensor, const char* name) {\n  TORCH_CHECK(tensor.is_cuda(), name, " must be a CUDA tensor");\n  TORCH_CHECK(tensor.is_contiguous(), name, " must be contiguous");\n}\n\nat::Tensor three_level_linear_cuda(\n    const at::Tensor& input, const at::Tensor& weight_int4,\n    const at::Tensor& scale_int4, const at::Tensor& zero_int4,\n    const at::Tensor& indices_int4, const at::Tensor& weight_int8,\n    const at::Tensor& scale_int8, const at::Tensor& indices_int8,\n    const at::Tensor& weight_fp16, const at::Tensor& indices_fp16,\n    int64_t group_size) {\n  check_cuda_contiguous(input, "input");\n  check_cuda_contiguous(weight_int4, "weight_int4");\n  check_cuda_contiguous(scale_int4, "scale_int4");\n  check_cuda_contiguous(zero_int4, "zero_int4");\n  check_cuda_contiguous(indices_int4, "indices_int4");\n  check_cuda_contiguous(weight_int8, "weight_int8");\n  check_cuda_contiguous(scale_int8, "scale_int8");\n  check_cuda_contiguous(indices_int8, "indices_int8");\n  check_cuda_contiguous(weight_fp16, "weight_fp16");\n  check_cuda_contiguous(indices_fp16, "indices_fp16");\n  TORCH_CHECK(input.dim() == 2, "input must have shape [rows, in_features]");\n  TORCH_CHECK(input.scalar_type() == at::kHalf ||\n                  input.scalar_type() == at::kFloat,\n              "input must be float16 or float32");\n  TORCH_CHECK(weight_int4.scalar_type() == at::kByte,\n              "weight_int4 must be uint8");\n  TORCH_CHECK(scale_int4.scalar_type() == at::kHalf,\n              "scale_int4 must be float16");\n  TORCH_CHECK(zero_int4.scalar_type() == at::kByte,\n              "zero_int4 must be uint8");\n  TORCH_CHECK(weight_int8.scalar_type() == at::kChar,\n              "weight_int8 must be int8");\n  TORCH_CHECK(scale_int8.scalar_type() == at::kHalf,\n              "scale_int8 must be float16");\n  TORCH_CHECK(weight_fp16.scalar_type() == at::kHalf,\n              "weight_fp16 must be float16");\n  TORCH_CHECK(indices_int4.scalar_type() == at::kInt &&\n                  indices_int8.scalar_type() == at::kInt &&\n                  indices_fp16.scalar_type() == at::kInt,\n              "all channel indices must be int32");\n  TORCH_CHECK(group_size > 0 && input.size(1) % group_size == 0,\n              "group_size must divide in_features");\n\n  const auto rows = input.size(0);\n  const auto width = input.size(1);\n  const auto n4 = indices_int4.numel();\n  const auto n8 = indices_int8.numel();\n  const auto n16 = indices_fp16.numel();\n  const auto output_width = n4 + n8 + n16;\n  TORCH_CHECK(output_width > 0, "at least one precision partition is required");\n  TORCH_CHECK(weight_int4.size(0) == n4 && weight_int4.size(1) == width / 2,\n              "invalid packed INT4 weight shape");\n  TORCH_CHECK(weight_int8.size(0) == n8 && weight_int8.size(1) == width,\n              "invalid INT8 weight shape");\n  TORCH_CHECK(weight_fp16.size(0) == n16 && weight_fp16.size(1) == width,\n              "invalid FP16 weight shape");\n  const auto groups = width / group_size;\n  TORCH_CHECK(scale_int4.size(0) == n4 && scale_int4.size(1) == groups &&\n                  zero_int4.size(0) == n4 && zero_int4.size(1) == groups,\n              "invalid INT4 scale/zero shape");\n  TORCH_CHECK(scale_int8.size(0) == n8 && scale_int8.size(1) == groups,\n              "invalid INT8 scale shape");\n\n  c10::cuda::CUDAGuard device_guard(input.device());\n  auto output = at::empty({rows, output_width}, input.options().dtype(at::kFloat));\n  const cudaStream_t stream = at::cuda::getCurrentCUDAStream();\n  const dim3 block(kThreads);\n  const dim3 grid(output_width, rows);\n\n  AT_DISPATCH_FLOATING_TYPES_AND_HALF(input.scalar_type(), "three_level_sm75", [&] {\n    three_level_partition_kernel<scalar_t><<<grid, block, 0, stream>>>(\n        input.data_ptr<scalar_t>(), weight_int4.data_ptr<uint8_t>(),\n        reinterpret_cast<const __half*>(scale_int4.data_ptr<at::Half>()),\n        zero_int4.data_ptr<uint8_t>(), indices_int4.data_ptr<int32_t>(),\n        weight_int8.data_ptr<int8_t>(),\n        reinterpret_cast<const __half*>(scale_int8.data_ptr<at::Half>()),\n        indices_int8.data_ptr<int32_t>(),\n        reinterpret_cast<const __half*>(weight_fp16.data_ptr<at::Half>()),\n        indices_fp16.data_ptr<int32_t>(), output.data_ptr<float>(), rows, width,\n        output_width, n4, n8, n16, group_size);\n  });\n  C10_CUDA_KERNEL_LAUNCH_CHECK();\n  return output;\n}\n\n}  // namespace\n\nTORCH_LIBRARY(mixllm_sm75, m) {\n  m.def("three_level_linear(Tensor input, Tensor weight_int4, Tensor scale_int4, "\n        "Tensor zero_int4, Tensor indices_int4, Tensor weight_int8, "\n        "Tensor scale_int8, Tensor indices_int8, Tensor weight_fp16, "\n        "Tensor indices_fp16, int group_size) -> Tensor");\n}\n\nTORCH_LIBRARY_IMPL(mixllm_sm75, CUDA, m) {\n  m.impl("three_level_linear", &three_level_linear_cuda);\n}', 'mixllm/test/test_three_level.py': 'import tempfile\nimport unittest\nfrom pathlib import Path\n\nimport torch\n\nfrom mixllm.quantization.three_level import (\n    ThreeLevelAllocation,\n    ThreeLevelBudget,\n    allocate_channels,\n    allocate_model_channels,\n    estimate_channel_losses,\n    fake_linear,\n)\nfrom mixllm.nn.modules.three_level_linear import ThreeLevelLinear\n\n\nclass ThreeLevelAllocationTest(unittest.TestCase):\n\n    def test_uses_marginal_benefit_for_each_upgrade(self):\n        losses = {\n            4: [100.0, 10.0, 9.0, 8.0],\n            8: [99.0, 0.0, 8.0, 7.0],\n            16: [0.0, 0.0, 8.0, 7.0],\n        }\n        result = allocate_channels(losses, ThreeLevelBudget(50, 25, 25))\n\n        self.assertEqual(result.indices[16], (0,))\n        self.assertEqual(result.indices[8], (1,))\n        self.assertEqual(result.indices[4], (2, 3))\n\n    def test_allows_empty_partitions(self):\n        losses = {4: [3.0, 2.0], 8: [1.0, 1.0], 16: [0.0, 0.0]}\n        result = allocate_channels(losses, ThreeLevelBudget(0, 0, 100))\n\n        self.assertEqual(result.indices[4], ())\n        self.assertEqual(result.indices[8], ())\n        self.assertEqual(result.indices[16], (0, 1))\n\n    def test_alignment_preserves_complete_partition(self):\n        losses = {bit: torch.arange(10, dtype=torch.float32) / bit for bit in (4, 8, 16)}\n        result = allocate_channels(losses, ThreeLevelBudget(60, 20, 20), alignment=2)\n\n        result.verify(10)\n        self.assertEqual({bit: len(result.indices[bit]) for bit in (4, 8, 16)},\n                         {4: 6, 8: 2, 16: 2})\n\n    def test_json_round_trip(self):\n        losses = {4: [3.0, 2.0], 8: [1.0, 1.0], 16: [0.0, 0.0]}\n        result = allocate_channels(losses, ThreeLevelBudget(50, 0, 50))\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / "allocation.json"\n            result.to_json(path)\n            loaded = ThreeLevelAllocation.from_json(path)\n\n        self.assertEqual(loaded, result)\n\n    def test_rejects_invalid_inputs(self):\n        with self.assertRaises(ValueError):\n            ThreeLevelBudget(90, 20, 0)\n        with self.assertRaises(ValueError):\n            allocate_channels({4: [], 8: [], 16: []}, ThreeLevelBudget(100, 0, 0))\n        with self.assertRaises(ValueError):\n            allocate_channels({4: [1], 8: [1, 2], 16: [0]},\n                              ThreeLevelBudget(100, 0, 0))\n\n    def test_global_allocator_preserves_counts_and_layer_minimums(self):\n        losses = {\n            "a": {4: [10, 9, 8, 7], 8: [1, 1, 1, 1], 16: [0, 0, 0, 0]},\n            "b": {4: [2, 2, 2, 2], 8: [1, 1, 1, 1], 16: [0, 0, 0, 0]},\n        }\n        result = allocate_model_channels(\n            losses, ThreeLevelBudget(50, 25, 25), layer_minimums={"b": 2})\n\n        counts = {bit: sum(len(value.indices[bit]) for value in result.values())\n                  for bit in (4, 8, 16)}\n        self.assertEqual(counts, {4: 4, 8: 2, 16: 2})\n        self.assertGreaterEqual(len(result["b"].indices[8]) +\n                                len(result["b"].indices[16]), 2)\n        for name, allocation in result.items():\n            allocation.verify(4)\n            self.assertEqual(allocation.metadata["layer_name"], name)\n\n\nclass ThreeLevelReferenceTest(unittest.TestCase):\n\n    def test_activation_aware_losses_share_reference_and_fp16_is_zero(self):\n        torch.manual_seed(5)\n        activation = torch.randn(2, 3, 128)\n        weight = torch.randn(6, 128)\n        losses, awq = estimate_channel_losses(activation, weight)\n\n        self.assertEqual(set(losses), {4, 8, 16})\n        torch.testing.assert_close(losses[16], torch.zeros(6), rtol=0, atol=0)\n        self.assertEqual(awq.shape, (128,))\n        self.assertTrue((losses[4] >= 0).all() and (losses[8] >= 0).all())\n\n    def test_all_fp16_is_exact(self):\n        torch.manual_seed(7)\n        x = torch.randn(3, 128, dtype=torch.float32)\n        weight = torch.randn(4, 128, dtype=torch.float32)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(0, 0, 100),\n        )\n\n        actual = fake_linear(x, weight, allocation)\n        expected = torch.nn.functional.linear(x, weight)\n        torch.testing.assert_close(actual, expected, rtol=0, atol=0)\n\n    def test_mixed_output_matches_per_partition_reference(self):\n        torch.manual_seed(11)\n        x = torch.randn(2, 128)\n        weight = torch.randn(4, 128)\n        allocation = ThreeLevelAllocation(\n            indices={4: (0, 1), 8: (2,), 16: (3,)},\n            scores={4: (0.0,) * 4, 8: (0.0,) * 4, 16: (0.0,) * 4},\n            budget=ThreeLevelBudget(50, 25, 25),\n        )\n\n        actual = fake_linear(x, weight, allocation)\n        self.assertEqual(actual.shape, (2, 4))\n        torch.testing.assert_close(actual[:, 3], x @ weight[3], rtol=0, atol=0)\n        self.assertTrue(torch.isfinite(actual).all())\n\n    def test_packed_module_matches_fake_reference(self):\n        torch.manual_seed(19)\n        x = torch.randn(3, 128)\n        weight = torch.randn(8, 128)\n        allocation = allocate_channels(\n            {4: torch.linspace(8, 1, 8),\n             8: torch.linspace(4, 0.5, 8),\n             16: torch.zeros(8)},\n            ThreeLevelBudget(50, 25, 25),\n        )\n        module = ThreeLevelLinear.from_weight(weight, allocation)\n\n        actual = module(x)\n        expected = fake_linear(x, weight, allocation)\n        torch.testing.assert_close(actual, expected, rtol=2e-3, atol=2e-3)\n\n    def test_packed_module_state_dict_round_trip(self):\n        weight = torch.randn(4, 128)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(50, 25, 25),\n        )\n        original = ThreeLevelLinear.from_weight(weight, allocation)\n        loaded = ThreeLevelLinear(128, 4)\n        loaded.load_state_dict(original.state_dict())\n\n        torch.testing.assert_close(loaded.dequantize_weight(), original.dequantize_weight())\n\n    def test_backward_compatible_two_level_state_dict(self):\n        weight = torch.randn(4, 128)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(50, 50, 0),\n        )\n        original = ThreeLevelLinear.from_weight(weight, allocation)\n        legacy = original.state_dict()\n        legacy.pop("weight_fp16")\n        legacy.pop("indices_16")\n        loaded = ThreeLevelLinear(128, 4)\n        loaded.load_state_dict(legacy, strict=True)\n\n        self.assertEqual(loaded.indices_16.numel(), 0)\n        torch.testing.assert_close(loaded.dequantize_weight(), original.dequantize_weight())\n\n\nif __name__ == "__main__":\n    unittest.main()', 'mixllm/test/test_runtime_capability.py': 'import unittest\n\nfrom mixllm.runtime_capability import RuntimeCapability\n\n\nclass RuntimeCapabilityTest(unittest.TestCase):\n\n    def test_t4_uses_reference_until_backend_is_validated(self):\n        self.assertEqual(RuntimeCapability(7, 5).select_backend(), "reference")\n\n    def test_auto_selects_validated_t4_backend(self):\n        self.assertEqual(RuntimeCapability(7, 5, sm75_available=True).select_backend(), "sm75")\n\n    def test_auto_selects_ampere_backend(self):\n        self.assertEqual(RuntimeCapability(8, 0).select_backend(), "ampere")\n\n    def test_unsupported_gpu_uses_reference(self):\n        self.assertEqual(RuntimeCapability(7, 0).select_backend(), "reference")\n\n    def test_rejects_forced_ampere_on_t4(self):\n        with self.assertRaises(RuntimeError):\n            RuntimeCapability(7, 5, "ampere").select_backend()\n\n    def test_rejects_unavailable_sm75_backend(self):\n        with self.assertRaises(RuntimeError):\n            RuntimeCapability(7, 5, "sm75").select_backend()\n\n    def test_rejects_unknown_backend(self):\n        with self.assertRaises(ValueError):\n            RuntimeCapability(8, 0, "unknown").select_backend()\n\n\nif __name__ == "__main__":\n    unittest.main()', 'mixllm/test/test_sm75_backend.py': 'import os\nimport unittest\n\nimport torch\n\nfrom mixllm.nn.modules.three_level_linear import ThreeLevelLinear\nfrom mixllm.quantization.three_level import ThreeLevelAllocation, ThreeLevelBudget\nfrom mixllm.sm75_backend import load_sm75_backend, three_level_linear\n\n\n@unittest.skipUnless(\n    os.environ.get("MIXLLM_TEST_SM75") == "1" and torch.cuda.is_available(),\n    "requires an explicit SM75 GPU test run",\n)\nclass SM75BackendTest(unittest.TestCase):\n\n    @classmethod\n    def setUpClass(cls):\n        if tuple(torch.cuda.get_device_capability()) != (7, 5):\n            raise unittest.SkipTest("requires compute capability 7.5")\n        load_sm75_backend(torch)\n\n    def _run_case(self, counts, rows=3, width=128, seed=31):\n        torch.manual_seed(seed)\n        n4, n8, n16 = counts\n        output_width = sum(counts)\n        order = torch.randperm(output_width).tolist()\n        allocation = ThreeLevelAllocation(\n            indices={\n                4: tuple(sorted(order[:n4])),\n                8: tuple(sorted(order[n4:n4 + n8])),\n                16: tuple(sorted(order[n4 + n8:])),\n            },\n            scores={bit: (0.0,) * output_width for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(100, 0, 0),\n        )\n        weight = torch.randn(output_width, width, device="cuda", dtype=torch.float16)\n        x = torch.randn(rows, width, device="cuda", dtype=torch.float16)\n        module = ThreeLevelLinear.from_weight(weight, allocation).cuda()\n\n        actual = three_level_linear(module, x, torch)\n        expected = torch.nn.functional.linear(x.float(), module.dequantize_weight())\n        torch.testing.assert_close(actual, expected, rtol=3e-3, atol=3e-3)\n        self.assertTrue(torch.isfinite(actual).all())\n\n    def test_mixed_and_empty_partitions(self):\n        for counts in ((5, 3, 2), (10, 0, 0), (0, 10, 0), (0, 0, 10),\n                       (0, 4, 6), (7, 0, 3)):\n            with self.subTest(counts=counts):\n                self._run_case(counts)\n\n    def test_random_rows_widths_and_determinism(self):\n        for rows, width in ((1, 128), (5, 128), (3, 256), (7, 384)):\n            with self.subTest(rows=rows, width=width):\n                self._run_case((4, 3, 3), rows=rows, width=width,\n                               seed=rows * 1000 + width)\n\n        allocation = ThreeLevelAllocation(\n            indices={4: (1,), 8: (2,), 16: (0,)},\n            scores={bit: (0.0,) * 3 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(100, 0, 0),\n        )\n        weight = torch.randn(3, 128, device="cuda", dtype=torch.float16)\n        x = torch.randn(3, 128, device="cuda", dtype=torch.float16)\n        module = ThreeLevelLinear.from_weight(weight, allocation).cuda()\n        first = three_level_linear(module, x, torch)\n        second = three_level_linear(module, x, torch)\n        torch.testing.assert_close(first, second, rtol=0, atol=0)\n\n    def test_non_default_stream_dependency(self):\n        allocation = ThreeLevelAllocation(\n            indices={4: (1,), 8: (2,), 16: (0,)},\n            scores={bit: (0.0,) * 3 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(34, 33, 33),\n        )\n        module = ThreeLevelLinear.from_weight(\n            torch.randn(3, 128, device="cuda", dtype=torch.float16), allocation,\n        ).cuda()\n        producer = torch.cuda.Stream()\n        consumer = torch.cuda.current_stream()\n        with torch.cuda.stream(producer):\n            x = torch.randn(5, 128, device="cuda", dtype=torch.float16)\n            ready = torch.cuda.Event()\n            ready.record()\n        consumer.wait_event(ready)\n        actual = three_level_linear(module, x, torch)\n        expected = torch.nn.functional.linear(x.float(), module.dequantize_weight())\n        torch.testing.assert_close(actual, expected, rtol=3e-3, atol=3e-3)\n\n    def test_cuda_graph_capture(self):\n        allocation = ThreeLevelAllocation(\n            indices={4: (0, 3), 8: (1,), 16: (2,)},\n            scores={bit: (0.0,) * 4 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(50, 25, 25),\n        )\n        module = ThreeLevelLinear.from_weight(\n            torch.randn(4, 128, device="cuda", dtype=torch.float16), allocation,\n        ).cuda()\n        static_input = torch.randn(2, 128, device="cuda", dtype=torch.float16)\n        side_stream = torch.cuda.Stream()\n        side_stream.wait_stream(torch.cuda.current_stream())\n        with torch.cuda.stream(side_stream):\n            for _ in range(3):\n                three_level_linear(module, static_input, torch)\n        torch.cuda.current_stream().wait_stream(side_stream)\n        graph = torch.cuda.CUDAGraph()\n        with torch.cuda.graph(graph):\n            captured = three_level_linear(module, static_input, torch)\n        graph.replay()\n        expected = torch.nn.functional.linear(static_input.float(), module.dequantize_weight())\n        torch.testing.assert_close(captured, expected, rtol=3e-3, atol=3e-3)\n\n\nif __name__ == "__main__":\n    unittest.main()', 'mixllm/test/test_vllm_three_level.py': 'import unittest\n\nfrom mixllm.vllm_three_level import (\n    PINNED_VLLM_COMMIT,\n    VLLMThreeLevelConfig,\n    remap_partition_indices_for_tp,\n    restore_global_partition_indices,\n    validate_partition_indices,\n)\n\n\nclass VLLMThreeLevelContractTest(unittest.TestCase):\n\n    def test_pins_real_upstream_submodule_commit(self):\n        self.assertEqual(PINNED_VLLM_COMMIT,\n                         "5fbbfe9a4c13094ad72ed3d6b4ef208a7ddc0fd7")\n\n    def test_config_and_backend_gate(self):\n        config = VLLMThreeLevelConfig.from_quantization_config({\n            "quant_method": "mixllm_three_level",\n            "precision_percentages": {"4": 75, "8": 20, "16": 5},\n            "group_size": 128,\n        })\n        self.assertEqual(config.select_backend((7, 5)), "reference")\n        self.assertEqual(config.select_backend((8, 0)), "ampere")\n\n    def test_requires_three_level_percentages(self):\n        with self.assertRaises(ValueError):\n            VLLMThreeLevelConfig.from_quantization_config({"quant_method": "mixllm"})\n\n    def test_partition_validation(self):\n        validate_partition_indices({4: [2, 3], 8: [1], 16: [0]}, 4)\n        with self.assertRaises(ValueError):\n            validate_partition_indices({4: [1], 8: [1], 16: [0]}, 3)\n\n    def test_tensor_parallel_index_remapping_round_trip(self):\n        global_indices = {4: [0, 3, 6, 7], 8: [1, 5], 16: [2, 4]}\n        local = remap_partition_indices_for_tp(global_indices, 8, 2, 4)\n        self.assertEqual(local, {4: (1,), 8: (3,), 16: (0, 2)})\n        restored = restore_global_partition_indices(local, 2, 4)\n        self.assertEqual(restored, {4: (3,), 8: (5,), 16: (2, 4)})\n\n    def test_tensor_parallel_rejects_invalid_shards(self):\n        indices = {4: [0, 1], 8: [2], 16: [3]}\n        with self.assertRaises(ValueError):\n            remap_partition_indices_for_tp(indices, 4, 3, 2)\n\n\nif __name__ == "__main__":\n    unittest.main()'}
root = Path('/kaggle/working/mixllm-3level')
for relative, text in sources.items():
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
sys.path.insert(0, str(root))
print('Embedded source files:', len(sources))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(root) + os.pathsep + test_env.get('PYTHONPATH', '')
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
if cap == (7, 5): test_env['MIXLLM_TEST_SM75'] = '1'
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(root / 'mixllm/test'), '-p', 'test_*.py', '-v'], cwd=root, env=test_env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0


In [ ]:
from mixllm.runtime_capability import RuntimeCapability
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
backend = RuntimeCapability(*cap).select_backend() if cap else 'reference'
is_t4 = cap == (7, 5) and gpu_name is not None and 'T4' in gpu_name
sm75_test_requested = test_env.get('MIXLLM_TEST_SM75') == '1'
report = {'schema_version': 3, 'capability': cap, 'gpu_name': gpu_name,
          'selected_production_backend': backend,
          'cuda_available': torch.cuda.is_available(),
          'reference_contract_gate': 'passed', 't4_hardware_gate': is_t4,
          'sm75_correctness_kernel_gate': ('passed' if sm75_test_requested else 'not_run'),
          'sm75_production_backend_ready': False,
          'native_three_level_gate': ('correctness_only' if sm75_test_requested else 'not_run'),
          'vllm_plugin_gate': 'contract_only',
          'model_quality_gate': 'not_run',
          'throughput_gate': 'not_run'}
Path('/kaggle/working/mixllm_3level_gate.json').write_text(json.dumps(report, indent=2))
print(report)
if cap == (7, 5):
    assert backend == 'reference'
    assert report['sm75_correctness_kernel_gate'] == 'passed'
print('MIXLLM THREE-LEVEL REFERENCE GATE: PASS')


In [ ]:
# Deterministic fake-quant and allocation gate, independent of CUDA kernels.
from mixllm.quantization.three_level import (ThreeLevelBudget, allocate_channels,
    estimate_channel_losses, allocate_model_channels)
torch.manual_seed(1234)
x = torch.randn(4, 3, 128)
w = torch.randn(8, 128)
losses, awq_stat = estimate_channel_losses(x, w)
allocation = allocate_channels(losses, ThreeLevelBudget(50, 25, 25))
allocation.verify(w.shape[0])
assert torch.isfinite(awq_stat).all() and torch.isfinite(losses[4]).all()
model_alloc = allocate_model_channels({'layer': losses}, ThreeLevelBudget(50, 25, 25))
assert sum(len(model_alloc['layer'].indices[b]) for b in (4, 8, 16)) == w.shape[0]
report['fake_quant_gate'] = 'passed'
print('FAKE QUANT / GLOBAL ALLOCATION GATE: PASS')


In [ ]:
# CUDA-event operator benchmark on the real SM75 implementation.
sm75_benchmark = {'status': 'not_run'}
if cap == (7, 5):
    from mixllm.nn.modules.three_level_linear import ThreeLevelLinear
    from mixllm.quantization.three_level import ThreeLevelAllocation, ThreeLevelBudget
    from mixllm.sm75_backend import benchmark_sm75_backend, load_sm75_backend
    load_sm75_backend(torch)
    n = 96; width = 512
    alloc = ThreeLevelAllocation(indices={4: tuple(range(0, 64)), 8: tuple(range(64, 88)), 16: tuple(range(88, 96))}, scores={b: (0.0,) * n for b in (4, 8, 16)}, budget=ThreeLevelBudget(67, 25, 8))
    packed = ThreeLevelLinear.from_weight(torch.randn(n, width, device='cuda', dtype=torch.float16), alloc).cuda()
    sm75_benchmark = benchmark_sm75_backend(packed, rows=(1, 8, 32, 128), torch_module=torch, warmup=10, iterations=50)
    errors_ok = all(shape['max_abs_error'] <= 0.05 for shape in sm75_benchmark['shapes'])
    performance_ok = all(shape['p50_ratio_vs_dense'] <= 1.05 for shape in sm75_benchmark['shapes'])
    report['sm75_graph_capture_gate'] = 'passed'
    report['sm75_production_backend_ready'] = bool(errors_ok and performance_ok)
    report['throughput_gate'] = 'operator_microbenchmark_measured'
report['sm75_operator_benchmark'] = sm75_benchmark
Path('/kaggle/working/mixllm_3level_gate.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
